# Linear 2D System: Nullcline Coalescence at det(A) = 0

For a linear system $\dot{\mathbf{x}} = A\mathbf{x}$, the nullclines are lines through the origin.
When $\det(A) = 0$, one eigenvalue becomes zero and the nullclines **coincide**,
creating a line of fixed points instead of a single fixed point at the origin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib widget

## System

$$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$

We fix $a = 1$, $b = 1$, $c = 1$ and vary $d$.

- $\det(A) = ad - bc = d - 1$
- **Critical value**: $d = 1$ gives $\det(A) = 0$

**Nullclines:**
- $\dot{x} = 0$: $ax + by = 0 \Rightarrow y = -x$ (red)
- $\dot{y} = 0$: $cx + dy = 0 \Rightarrow y = -x/d$ (blue)

When $d = 1$, both nullclines are $y = -x$ — they coincide!

In [ ]:
# Fixed matrix entries
a, b, c = 1.0, 1.0, 1.0

def get_matrix(d):
    return np.array([[a, b], [c, d]])

def vector_field(x, y, d):
    """Compute dx/dt, dy/dt for the linear system."""
    dx = a * x + b * y
    dy = c * x + d * y
    return dx, dy

def get_eigenvalues(d):
    A = get_matrix(d)
    return np.linalg.eigvals(A)

In [ ]:
def plot_linear_system(d=0.5):
    """Plot the linear system with nullclines and eigenvalues."""
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    
    A = get_matrix(d)
    det_A = np.linalg.det(A)
    tr_A = np.trace(A)
    eigs = get_eigenvalues(d)
    
    # === Left panel: Phase portrait ===
    ax = axes[0]
    
    # Vector field grid
    x_range = np.linspace(-2, 2, 15)
    y_range = np.linspace(-2, 2, 15)
    X, Y = np.meshgrid(x_range, y_range)
    DX, DY = vector_field(X, Y, d)
    
    # Normalize arrows
    mag = np.sqrt(DX**2 + DY**2)
    mag[mag == 0] = 1
    
    ax.quiver(X, Y, DX/mag, DY/mag, mag, cmap='viridis', alpha=0.7, scale=25)
    
    # Nullclines
    x_line = np.linspace(-2.5, 2.5, 100)
    
    # x-nullcline: ax + by = 0 → y = -(a/b)x = -x
    y_null_x = -(a/b) * x_line
    ax.plot(x_line, y_null_x, 'r-', linewidth=3, label=r'$\dot{x}=0$: $y = -x$')
    
    # y-nullcline: cx + dy = 0 → y = -(c/d)x
    if np.abs(d) > 1e-6:
        y_null_y = -(c/d) * x_line
        ax.plot(x_line, y_null_y, 'b-', linewidth=3, 
                label=rf'$\dot{{y}}=0$: $y = -{c/d:.2f}x$')
    else:
        # d ≈ 0: y-nullcline is x = 0 (vertical line)
        ax.axvline(0, color='blue', linewidth=3, label=r'$\dot{y}=0$: $x = 0$')
    
    # Fixed point at origin (or line of fixed points)
    if np.abs(det_A) < 0.05:
        # Line of fixed points along the nullcline
        ax.plot(x_line, y_null_x, 'ko', markersize=3, alpha=0.5)
        ax.plot(0, 0, 'ko', markersize=12, markeredgewidth=2, 
                markerfacecolor='yellow', label='Line of fixed points')
    else:
        ax.plot(0, 0, 'ko', markersize=12, markeredgewidth=2,
                markerfacecolor='white', label='Fixed point')
    
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    
    # Title with system info
    fp_type = 'SADDLE' if det_A < 0 else ('SINGULAR' if np.abs(det_A) < 0.05 else 'NODE')
    ax.set_title(f'd = {d:.2f}\ndet(A) = {det_A:.2f}, tr(A) = {tr_A:.2f}\nType: {fp_type}', 
                 fontsize=12)
    
    # === Right panel: Eigenvalues ===
    ax2 = axes[1]
    
    eig_real = np.real(eigs)
    eig_imag = np.imag(eigs)
    
    # Bar chart for real parts
    x_pos = [0, 1]
    colors = ['tab:blue', 'tab:orange']
    
    bars = ax2.bar(x_pos, eig_real, width=0.6, color=colors, 
                   edgecolor='black', linewidth=2)
    
    # Add value labels
    for i, (bar, er, ei) in enumerate(zip(bars, eig_real, eig_imag)):
        height = bar.get_height()
        va = 'bottom' if height >= 0 else 'top'
        offset = 0.1 if height >= 0 else -0.1
        
        if np.abs(ei) < 1e-6:
            label = f'{er:.3f}'
        else:
            label = f'{er:.2f}±{np.abs(ei):.2f}i'
        
        ax2.text(bar.get_x() + bar.get_width()/2., height + offset,
                label, ha='center', va=va, fontsize=12, fontweight='bold')
    
    # Zero line (critical)
    ax2.axhline(0, color='red', linewidth=3, linestyle='--', 
                label='λ = 0 (singular)')
    
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['λ₁', 'λ₂'], fontsize=14)
    ax2.set_ylabel('Eigenvalue (real part)', fontsize=12)
    ax2.set_ylim(-3, 3)
    ax2.set_title('Eigenvalues of A', fontsize=12)
    ax2.legend(loc='upper right', fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Color background based on det(A)
    if det_A < -0.05:
        ax2.set_facecolor('#fff0f0')  # Light red for saddle
    elif det_A > 0.05:
        ax2.set_facecolor('#f0fff0')  # Light green for node
    else:
        ax2.set_facecolor('#ffffd0')  # Yellow for singular
    
    plt.tight_layout()
    plt.show()
    
    return fig

## Interactive Visualization

Slide $d$ to cross the critical value $d = 1$ where $\det(A) = 0$:

- **d < 1**: Saddle (det < 0, eigenvalues have opposite signs, nullclines intersect at origin)
- **d = 1**: Singular (det = 0, one eigenvalue = 0, nullclines coincide → line of fixed points)
- **d > 1**: Node (det > 0, eigenvalues have same sign)

In [ ]:
interact(
    plot_linear_system,
    d=FloatSlider(
        value=0.5,
        min=-1.0,
        max=3.0,
        step=0.02,
        description='d:',
        continuous_update=False,
        readout_format='.2f',
        style={'description_width': '30px'},
        layout={'width': '500px'}
    )
);

## Static Comparison: Before, At, and After det(A) = 0

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

d_values = [0.5, 1.0, 1.5]
titles = ['Saddle (d=0.5, det<0)', 'Singular (d=1, det=0)', 'Node (d=1.5, det>0)']

for col, (d, title) in enumerate(zip(d_values, titles)):
    
    A = get_matrix(d)
    det_A = np.linalg.det(A)
    eigs = get_eigenvalues(d)
    
    # === Top row: Phase portrait ===
    ax = axes[0, col]
    
    x_range = np.linspace(-2, 2, 12)
    y_range = np.linspace(-2, 2, 12)
    X, Y = np.meshgrid(x_range, y_range)
    DX, DY = vector_field(X, Y, d)
    
    mag = np.sqrt(DX**2 + DY**2)
    mag[mag == 0] = 1
    
    ax.quiver(X, Y, DX/mag, DY/mag, mag, cmap='viridis', alpha=0.7, scale=22)
    
    # Nullclines
    x_line = np.linspace(-2.5, 2.5, 100)
    y_null_x = -x_line  # y = -x
    y_null_y = -(1/d) * x_line if d != 0 else None
    
    ax.plot(x_line, y_null_x, 'r-', linewidth=3, label=r'$\dot{x}=0$')
    if y_null_y is not None:
        ax.plot(x_line, y_null_y, 'b--', linewidth=3, label=r'$\dot{y}=0$')
    
    # Fixed point
    if np.abs(det_A) < 0.01:
        ax.plot(x_line, y_null_x, 'ko', markersize=2, alpha=0.3)
    ax.plot(0, 0, 'ko', markersize=10, markerfacecolor='white', markeredgewidth=2)
    
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)
    
    # === Bottom row: Eigenvalues ===
    ax2 = axes[1, col]
    
    eig_real = np.real(eigs)
    bars = ax2.bar([0, 1], eig_real, color=['tab:blue', 'tab:orange'], 
                   edgecolor='black', width=0.6)
    
    for bar, er in zip(bars, eig_real):
        height = bar.get_height()
        va = 'bottom' if height >= 0 else 'top'
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1 * np.sign(height),
                f'{er:.2f}', ha='center', va=va, fontsize=11, fontweight='bold')
    
    ax2.axhline(0, color='red', linewidth=2, linestyle='--')
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels(['λ₁', 'λ₂'], fontsize=12)
    ax2.set_ylabel('Eigenvalue')
    ax2.set_ylim(-2.5, 3)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_title(f'det(A) = {det_A:.2f}', fontsize=10)

plt.suptitle(r'Linear System $\dot{\mathbf{x}} = A\mathbf{x}$: Nullcline Coalescence at det(A)=0', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()